<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/ROSG_Test5_V3b_IFS_conductance_point_reconstruction_i1_i5_scaled_lambda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG Test5 V3b — IFS conductance-point reconstruction

**Mode:** Codex + autotests + checkpoint/cache by `level_i`.

New in V3b: adds the scaled spectral diagnostic $\lambda_{1,scaled}^{(i)}=4^i\lambda_1^{(i)}$ to factor out dyadic mesh refinement.

This notebook does **not** claim DSI detection; it prepares Test6.

## Imports, configuration and output directory

In [1]:
# ============================================================
# ROSG Test5 V3b — IFS conductance-point reconstruction
# Checkpoint/cache by level_i, with i=1..5
# ------------------------------------------------------------
# Purpose:
#   Reconstruct the IFS/PCF support carrying effective conductances.
#   This is NOT a DSI detection test. It prepares Test6.
# ============================================================

import os, json, math, hashlib, zipfile, shutil, time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import scipy.sparse as sp
    from scipy.sparse.csgraph import connected_components
    from scipy.sparse.linalg import eigsh
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False


# ---------------------------
# Configuration
# ---------------------------

@dataclass
class Test5V3bConfig:
    project_name: str = "ROSG_Test5_V3b_IFS_conductance_point_reconstruction_i1_i5_scaled_lambda"
    levels_i: tuple = (1, 2, 3, 4, 5)
    stabilization_levels_i: tuple = (3, 4, 5)
    calibration_levels_i: tuple = (1, 2)
    lambda_lattice: float = 2.0
    base_conductance: float = 1.0
    vertical_gain: float = 2.1
    activation_center_Z: float = 1.75
    activation_width_Z: float = 0.75
    Z_min: float = -1.0
    Z_max: float = 5.0
    n_Z: int = 17
    random_seed: int = 20260604
    out_dir: str = "/content/ROSG_Test5_V3b_IFS_conductance_point_reconstruction_i1_i5_scaled_lambda"
    use_google_drive: bool = True
    drive_dir: str = "/content/drive/MyDrive/ROSG_exports/ROSG_Test5_V3b_IFS_conductance_point_reconstruction_i1_i5_scaled_lambda"
    force_recompute: bool = False

CFG = Test5V3bConfig()


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def prepare_output_dir(cfg=CFG):
    out = Path(cfg.out_dir)
    if cfg.use_google_drive and in_colab():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive", force_remount=False)
            if Path("/content/drive/MyDrive").exists():
                out = Path(cfg.drive_dir)
        except Exception as exc:
            print("[Drive warning]", exc)
            out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    return out


OUT_DIR = prepare_output_dir(CFG)
print("Output directory:", OUT_DIR)
print("Levels i:", CFG.levels_i)
print("Calibration levels:", CFG.calibration_levels_i)
print("Stabilization levels:", CFG.stabilization_levels_i)

Mounted at /content/drive
Output directory: /content/drive/MyDrive/ROSG_exports/ROSG_Test5_V3b_IFS_conductance_point_reconstruction_i1_i5_scaled_lambda
Levels i: (1, 2, 3, 4, 5)
Calibration levels: (1, 2)
Stabilization levels: (3, 4, 5)


## IFS / PCF reconstruction

In [2]:
# ---------------------------
# IFS / PCF reconstruction
# ---------------------------

def level_role(i, cfg=CFG):
    if i in cfg.calibration_levels_i:
        return "calibration_control"
    if i in cfg.stabilization_levels_i:
        return "stabilization_numeric"
    return "auxiliary"


def grid_size_from_i(i):
    return 2 ** int(i)


def build_Vi(i):
    """Return IFS collage point set V_i for a square cell-4 dyadic IFS."""
    L = grid_size_from_i(i)
    pts = []
    for y in range(L + 1):
        for x in range(L + 1):
            pts.append({
                "point_id": y * (L + 1) + x,
                "i": int(i),
                "x_index": int(x),
                "y_index": int(y),
                "x": float(x / L),
                "y": float(y / L),
                "is_boundary": bool(x == 0 or y == 0 or x == L or y == L),
                "role": level_role(i),
            })
    return pd.DataFrame(pts)


def build_Ei(i):
    """Return approximation graph E_i on the dyadic IFS collage set V_i."""
    L = grid_size_from_i(i)
    rows = []
    edge_id = 0

    def node_id(x, y):
        return y * (L + 1) + x

    for y in range(L + 1):
        for x in range(L + 1):
            u = node_id(x, y)
            if x < L:
                v = node_id(x + 1, y)
                rows.append({"edge_id": edge_id, "i": i, "u": u, "v": v, "orientation": "horizontal", "role": level_role(i)})
                edge_id += 1
            if y < L:
                v = node_id(x, y + 1)
                rows.append({"edge_id": edge_id, "i": i, "u": u, "v": v, "orientation": "vertical", "role": level_role(i)})
                edge_id += 1
    return pd.DataFrame(rows)


def local_hierarchy_level_for_edge(row, i):
    """Proxy for dyadic hierarchy on an edge midpoint."""
    L = grid_size_from_i(i)
    u, v = int(row["u"]), int(row["v"])
    ux, uy = u % (L + 1), u // (L + 1)
    vx, vy = v % (L + 1), v // (L + 1)
    mx2 = ux + vx  # twice midpoint in grid units
    my2 = uy + vy
    # count divisibility by 2 in midpoint coordinates, bounded by i
    def tz(a):
        a = int(abs(a))
        if a == 0:
            return int(i)
        c = 0
        while a % 2 == 0 and c < i:
            c += 1
            a //= 2
        return c
    return int(min(i, max(tz(mx2), tz(my2))))


def activation_multiplicity(Z, theta, width):
    """Smooth activation of unresolved microcontacts into effective multiplicity."""
    return float(1.0 / (1.0 + math.exp(-(float(Z) - float(theta)) / max(float(width), 1e-12))))


def conductance_for_edge(row, Z, cfg=CFG):
    h = local_hierarchy_level_for_edge(row, int(row["i"]))
    # hierarchy-aware threshold: deeper dyadic junctions activate slightly earlier
    theta = cfg.activation_center_Z - 0.12 * h
    m = activation_multiplicity(Z, theta, cfg.activation_width_Z)
    c = cfg.base_conductance + cfg.vertical_gain * m
    return m, c, theta, h


def annotate_edges_with_conductance(Ei, Z, cfg=CFG):
    rows = []
    for _, row in Ei.iterrows():
        m, c, theta, h = conductance_for_edge(row, Z, cfg)
        d = dict(row)
        d.update({
            "Z": float(Z),
            "hierarchy_h": int(h),
            "theta_edge": float(theta),
            "m_eff": float(m),
            "c_eff": float(c),
        })
        rows.append(d)
    return pd.DataFrame(rows)


def graph_stats(Vi, Ei):
    n = len(Vi)
    degrees = np.zeros(n, dtype=int)
    for _, e in Ei.iterrows():
        degrees[int(e["u"])] += 1
        degrees[int(e["v"])] += 1
    max_incidence = int(degrees.max()) if len(degrees) else 0

    if SCIPY_AVAILABLE:
        data = np.ones(2 * len(Ei))
        rows = []
        cols = []
        for _, e in Ei.iterrows():
            u, v = int(e["u"]), int(e["v"])
            rows += [u, v]
            cols += [v, u]
        A = sp.coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()
        n_components, labels = connected_components(A, directed=False)
        counts = np.bincount(labels)
        lcc_ratio = float(counts.max() / n) if len(counts) else 0.0
    else:
        # deterministic for the square grid here
        n_components, lcc_ratio = 1, 1.0

    L = grid_size_from_i(int(Vi["i"].iloc[0]))
    return {
        "num_points_Vi": int(len(Vi)),
        "num_edges_Ei": int(len(Ei)),
        "num_cells": int(4 ** int(Vi["i"].iloc[0])),
        "grid_L": int(L),
        "max_incidence": int(max_incidence),
        "n_components": int(n_components),
        "lcc_ratio": float(lcc_ratio),
    }


def weighted_laplacian_from_edges(n_points, Ec):
    if not SCIPY_AVAILABLE:
        return None
    rows, cols, data = [], [], []
    deg = np.zeros(n_points, dtype=float)
    for _, e in Ec.iterrows():
        u, v, c = int(e["u"]), int(e["v"]), float(e["c_eff"])
        rows += [u, v]
        cols += [v, u]
        data += [-c, -c]
        deg[u] += c
        deg[v] += c
    rows += list(range(n_points))
    cols += list(range(n_points))
    data += list(deg)
    return sp.coo_matrix((data, (rows, cols)), shape=(n_points, n_points)).tocsr()


def lambda1_laplacian(n_points, Ec):
    L = weighted_laplacian_from_edges(n_points, Ec)
    if L is None:
        return float("nan")
    try:
        if n_points <= 3:
            vals = np.linalg.eigvalsh(L.toarray())
            vals = np.sort(vals)
            return float(vals[1]) if len(vals) > 1 else float("nan")
        k = min(3, n_points - 1)
        vals = eigsh(L, k=k, which="SM", return_eigenvectors=False)
        vals = np.sort(np.real(vals))
        # first non-zero
        nz = vals[vals > 1e-9]
        return float(nz[0]) if len(nz) else float("nan")
    except Exception:
        try:
            vals = np.linalg.eigvalsh(L.toarray())
            vals = np.sort(vals)
            nz = vals[vals > 1e-9]
            return float(nz[0]) if len(nz) else float("nan")
        except Exception:
            return float("nan")


def scale_lambda1(lambda1, i):
    """Dyadic mesh h_i = 2^-i, so h_i^-2 = 4^i."""
    return float(lambda1 * (4 ** int(i)))

## Cache/checkpoint by level_i

In [3]:
# ---------------------------
# Cache/checkpoint by level_i
# ---------------------------

def level_dir(i):
    d = OUT_DIR / f"level_i_{int(i)}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def required_level_files(i):
    d = level_dir(i)
    return [
        d / "ifs_points_Vi.csv",
        d / "ifs_edges_Ei.csv",
        d / f"level_i_{int(i)}_scan.csv",
        d / f"level_i_{int(i)}_summary.json",
        d / f"level_i_{int(i)}_config.json",
        d / "COMPLETE.flag",
    ]


def is_level_complete(i):
    return all(p.exists() and p.stat().st_size > 0 for p in required_level_files(i))


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def compute_level(i, cfg=CFG):
    d = level_dir(i)
    if is_level_complete(i) and not cfg.force_recompute:
        print(f"[cache] level_i={i} already complete; loading.")
        scan = pd.read_csv(d / f"level_i_{int(i)}_scan.csv")
        summary = load_json(d / f"level_i_{int(i)}_summary.json")
        return scan, summary

    print(f"[compute] level_i={i}")
    Vi = build_Vi(i)
    Ei = build_Ei(i)
    stats = graph_stats(Vi, Ei)

    Z_grid = np.linspace(cfg.Z_min, cfg.Z_max, cfg.n_Z)
    scan_rows = []
    for Z in Z_grid:
        Ec = annotate_edges_with_conductance(Ei, Z, cfg)
        lam1 = lambda1_laplacian(len(Vi), Ec)
        lam1_scaled = scale_lambda1(lam1, i)
        scan_rows.append({
            "i": int(i),
            "role": level_role(i),
            "Z": float(Z),
            **stats,
            "m_eff_mean": float(Ec["m_eff"].mean()),
            "m_eff_std": float(Ec["m_eff"].std(ddof=0)),
            "c_eff_mean": float(Ec["c_eff"].mean()),
            "c_eff_std": float(Ec["c_eff"].std(ddof=0)),
            "lambda1_weighted": float(lam1),
            "lambda1_scaled_4powi": float(lam1_scaled),
        })

    scan = pd.DataFrame(scan_rows)
    low = scan.head(3).mean(numeric_only=True)
    high = scan.tail(3).mean(numeric_only=True)
    omega0 = float(2.0 * math.pi / math.log(cfg.lambda_lattice))

    summary = {
        "i": int(i),
        "role": level_role(i),
        **stats,
        "Z_min": float(cfg.Z_min),
        "Z_max": float(cfg.Z_max),
        "n_Z": int(cfg.n_Z),
        "m_low_first3_mean": float(low["m_eff_mean"]),
        "m_high_last3_mean": float(high["m_eff_mean"]),
        "m_gain": float(high["m_eff_mean"] - low["m_eff_mean"]),
        "c_low_first3_mean": float(low["c_eff_mean"]),
        "c_high_last3_mean": float(high["c_eff_mean"]),
        "c_gain": float(high["c_eff_mean"] - low["c_eff_mean"]),
        "lambda1_low_first3_mean": float(low["lambda1_weighted"]),
        "lambda1_high_last3_mean": float(high["lambda1_weighted"]),
        "lambda1_scaled_low_first3_mean": float(low["lambda1_scaled_4powi"]),
        "lambda1_scaled_high_last3_mean": float(high["lambda1_scaled_4powi"]),
        "lattice_candidate": True,
        "lambda_lattice": float(cfg.lambda_lattice),
        "omega0_for_Test6": omega0,
        "status": "completed",
    }

    Vi.to_csv(d / "ifs_points_Vi.csv", index=False)
    Ei.to_csv(d / "ifs_edges_Ei.csv", index=False)
    scan.to_csv(d / f"level_i_{int(i)}_scan.csv", index=False)
    save_json(summary, d / f"level_i_{int(i)}_summary.json")
    save_json(asdict(cfg), d / f"level_i_{int(i)}_config.json")
    (d / "COMPLETE.flag").write_text(time.strftime("%Y-%m-%d %H:%M:%S"), encoding="utf-8")
    return scan, summary


def run_all_levels(cfg=CFG):
    all_scans = []
    summaries = []
    for i in cfg.levels_i:
        scan, summary = compute_level(i, cfg)
        all_scans.append(scan)
        summaries.append(summary)

    cross = pd.DataFrame(summaries).sort_values("i").reset_index(drop=True)
    scan_all = pd.concat(all_scans, ignore_index=True)

    cross.to_csv(OUT_DIR / "cross_level_comparison.csv", index=False)
    scan_all.to_csv(OUT_DIR / "all_levels_scan.csv", index=False)

    # Stabilization metrics only on i>=3
    stab = cross[cross["i"].isin(cfg.stabilization_levels_i)].copy()
    metrics = {}
    for col in ["m_gain", "c_gain", "c_high_last3_mean", "lambda1_high_last3_mean", "lambda1_scaled_high_last3_mean"]:
        vals = stab[col].to_numpy(dtype=float)
        metrics[col + "_mean_stabilization"] = float(np.nanmean(vals))
        metrics[col + "_std_stabilization"] = float(np.nanstd(vals))
        metrics[col + "_rel_std_stabilization"] = float(np.nanstd(vals) / max(abs(np.nanmean(vals)), 1e-12))
    metrics["levels_used_for_stabilization"] = list(map(int, cfg.stabilization_levels_i))
    metrics["levels_used_for_calibration"] = list(map(int, cfg.calibration_levels_i))
    metrics["status"] = "completed"

    pd.DataFrame([metrics]).to_csv(OUT_DIR / "cross_level_stabilization_metrics.csv", index=False)

    run_index = {
        "project": cfg.project_name,
        "levels_i": list(map(int, cfg.levels_i)),
        "calibration_levels_i": list(map(int, cfg.calibration_levels_i)),
        "stabilization_levels_i": list(map(int, cfg.stabilization_levels_i)),
        "completed_levels_i": [int(x) for x in cross["i"].tolist()],
        "output_dir": str(OUT_DIR),
        "status": "completed",
    }
    save_json(run_index, OUT_DIR / "RUN_INDEX.json")

    report = {
        "status": "completed",
        "purpose": "IFS/PCF reconstruction of effective conductance-carrying collage points and approximation graphs, with scaled spectral diagnostic.",
        "new_in_V3b": "Adds lambda1_scaled_4powi = 4**i * lambda1_weighted to factor out dyadic mesh refinement.",
        "not_a_DSI_detection": True,
        "interpretation": (
            "Levels i=1,2 are calibration/control levels. "
            "Levels i=3,4,5 are used for numerical stabilization diagnostics. "
            "The lattice frequency omega0 is reported only as preparation for Test6."
        ),
        "run_index": run_index,
        "stabilization_metrics": metrics,
    }
    save_json(report, OUT_DIR / "test5_v3_report.json")

    return cross, scan_all, metrics, report

## Figures and export

In [4]:
def make_figures():
    cross = pd.read_csv(OUT_DIR / "cross_level_comparison.csv")
    scan_all = pd.read_csv(OUT_DIR / "all_levels_scan.csv")

    # Figure 1: activation by level
    plt.figure(figsize=(8, 5))
    for i, g in scan_all.groupby("i"):
        plt.plot(g["Z"], g["m_eff_mean"], marker="o", label=f"i={i}")
    plt.xlabel("Z")
    plt.ylabel("mean effective multiplicity m_eff")
    plt.title("Test5 V3 — activation by IFS iteration level i")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "activation_by_level_i.png", dpi=160)
    plt.close()

    # Figure 2: conductance by level
    plt.figure(figsize=(8, 5))
    for i, g in scan_all.groupby("i"):
        plt.plot(g["Z"], g["c_eff_mean"], marker="o", label=f"i={i}")
    plt.xlabel("Z")
    plt.ylabel("mean effective conductance c_eff")
    plt.title("Test5 V3 — conductance by IFS iteration level i")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "conductance_by_level_i.png", dpi=160)
    plt.close()

    # Figure 3: lambda1 by level
    plt.figure(figsize=(8, 5))
    for i, g in scan_all.groupby("i"):
        plt.plot(g["Z"], g["lambda1_weighted"], marker="o", label=f"i={i}")
    plt.xlabel("Z")
    plt.ylabel("first non-zero weighted Laplacian eigenvalue")
    plt.title("Test5 V3 — lambda1 by IFS iteration level i")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "lambda1_by_level_i.png", dpi=160)
    plt.close()

    # Figure 4: scaled lambda1 by level
    plt.figure(figsize=(8, 5))
    for i, g in scan_all.groupby("i"):
        plt.plot(g["Z"], g["lambda1_scaled_4powi"], marker="o", label=f"i={i}")
    plt.xlabel("Z")
    plt.ylabel("lambda1_scaled = 4^i lambda1")
    plt.title("Test5 V3b — scaled lambda1 by IFS iteration level i")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "lambda1_scaled_by_level_i.png", dpi=160)
    plt.close()

    # Figure 4: counts
    plt.figure(figsize=(8, 5))
    plt.plot(cross["i"], cross["num_points_Vi"], marker="o", label="|V_i|")
    plt.plot(cross["i"], cross["num_edges_Ei"], marker="o", label="|E_i|")
    plt.xlabel("IFS iteration level i")
    plt.ylabel("count")
    plt.title("Test5 V3 — approximation graph size by level i")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "graph_size_by_level_i.png", dpi=160)
    plt.close()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def write_readme_manifest_and_zip(cfg=CFG):
    readme_lines = [
        "# ROSG Test5 V3b — IFS conductance-point reconstruction",
        "",
        "This archive contains the Test5 V3b outputs.",
        "",
        "Purpose:",
        "- reconstruct the IFS/PCF approximation supports V_i and E_i;",
        "- attach effective conductances c_xy^(i)(Z) and multiplicities m_xy^(i)(Z);",
        "- preserve the study notation using the iteration index i;",
        "- add lambda1_scaled_4powi = 4^i * lambda1_weighted;",
        "- include i=1,2 as calibration/control levels;",
        "- use i=3,4,5 for stabilization diagnostics;",
        "- prepare, but not execute, the future Test6 lattice/non-lattice DSI residual audit.",
        "",
        "Important:",
        "This test does not claim DSI detection.",
        "It reconstructs the conductance-carrying IFS support required before a DSI test.",
        "",
        "Main files:",
        "- cross_level_comparison.csv",
        "- cross_level_stabilization_metrics.csv",
        "- all_levels_scan.csv",
        "- RUN_INDEX.json",
        "- test5_v3_report.json",
        "- level_i_*/ifs_points_Vi.csv",
        "- level_i_*/ifs_edges_Ei.csv",
        "- level_i_*/level_i_*_scan.csv",
        "- level_i_*/level_i_*_summary.json",
        "",
    ]
    (OUT_DIR / "README.md").write_text("\n".join(readme_lines), encoding="utf-8")

    rows = []
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file() and p.name != "manifest_sha256.csv":
            rows.append({
                "path": str(p.relative_to(OUT_DIR)),
                "size_bytes": int(p.stat().st_size),
                "sha256": sha256_file(p),
            })
    manifest = pd.DataFrame(rows)
    manifest.to_csv(OUT_DIR / "manifest_sha256.csv", index=False)

    zip_path = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    print("ZIP:", zip_path)
    return zip_path

## Autotests

In [5]:
def run_autotests():
    print("Running Test5 V3b autotests...")
    for i in CFG.levels_i:
        Vi = build_Vi(i)
        Ei = build_Ei(i)
        st = graph_stats(Vi, Ei)
        L = grid_size_from_i(i)
        assert len(Vi) == (L + 1) ** 2
        assert len(Ei) == 2 * L * (L + 1)
        assert st["max_incidence"] <= 4
        assert st["n_components"] == 1
        assert abs(st["lcc_ratio"] - 1.0) < 1e-12
    assert 1 in CFG.calibration_levels_i
    assert 2 in CFG.calibration_levels_i
    assert 3 in CFG.stabilization_levels_i
    assert 4 in CFG.stabilization_levels_i
    assert 5 in CFG.stabilization_levels_i
    assert abs(scale_lambda1(1.0, 3) - 64.0) < 1e-12
    print("All autotests passed.")

## Run Test5 V3b

In [6]:
run_autotests()
cross, scan_all, metrics, report = run_all_levels(CFG)
make_figures()
zip_path = write_readme_manifest_and_zip(CFG)
print(json.dumps({
    "status": "completed",
    "output_dir": str(OUT_DIR),
    "zip_path": zip_path,
    "levels_i": list(map(int, CFG.levels_i)),
    "calibration_levels_i": list(map(int, CFG.calibration_levels_i)),
    "stabilization_levels_i": list(map(int, CFG.stabilization_levels_i)),
    "lambda1_scaled_added": True,
}, indent=2))
display(cross)
display(pd.DataFrame([metrics]))


Running Test5 V3 autotests...
All autotests passed.
[compute] level_i=1
[compute] level_i=2
[compute] level_i=3
[compute] level_i=4
[compute] level_i=5
ZIP: /content/drive/MyDrive/ROSG_exports/ROSG_Test5_V3b_IFS_conductance_point_reconstruction_i1_i5_scaled_lambda.zip
{
  "status": "completed",
  "output_dir": "/content/drive/MyDrive/ROSG_exports/ROSG_Test5_V3b_IFS_conductance_point_reconstruction_i1_i5_scaled_lambda",
  "zip_path": "/content/drive/MyDrive/ROSG_exports/ROSG_Test5_V3b_IFS_conductance_point_reconstruction_i1_i5_scaled_lambda.zip",
  "levels_i": [
    1,
    2,
    3,
    4,
    5
  ],
  "calibration_levels_i": [
    1,
    2
  ],
  "stabilization_levels_i": [
    3,
    4,
    5
  ],
  "lambda1_scaled_added": true
}


,i,role,num_points_Vi,num_edges_Ei,num_cells,grid_L,max_incidence,n_components,lcc_ratio,Z_min,...,c_high_last3_mean,c_gain,lambda1_low_first3_mean,lambda1_high_last3_mean,lambda1_scaled_low_first3_mean,lambda1_scaled_high_last3_mean,lattice_candidate,lambda_lattice,omega0_for_Test6,status
0,1,calibration_control,9,12,4,2,4,1,1.0,-1.0,...,3.058933,1.952785,1.106149,3.058933,4.424594,12.235734,True,2.0,9.06472,completed
1,2,calibration_control,25,40,16,4,4,1,1.0,-1.0,...,3.062506,1.946047,0.426446,1.169773,6.823136,18.716372,True,2.0,9.06472,completed
2,3,stabilization_numeric,81,144,64,8,4,1,1.0,-1.0,...,3.063944,1.941669,0.135361,0.369557,8.663110,23.651634,True,2.0,9.06472,completed
3,4,stabilization_numeric,289,544,256,16,4,1,1.0,-1.0,...,3.064360,1.939319,0.038311,0.104353,9.807735,26.714390,True,2.0,9.06472,completed
4,5,stabilization_numeric,1089,2112,1024,32,4,1,1.0,-1.0,...,3.064397,1.938244,0.010199,0.027752,10.443310,28.417691,True,2.0,9.06472,completed


,m_gain_mean_stabilization,m_gain_std_stabilization,m_gain_rel_std_stabilization,c_gain_mean_stabilization,c_gain_std_stabilization,c_gain_rel_std_stabilization,c_high_last3_mean_mean_stabilization,c_high_last3_mean_std_stabilization,c_high_last3_mean_rel_std_stabilization,lambda1_high_last3_mean_mean_stabilization,lambda1_high_last3_mean_std_stabilization,lambda1_high_last3_mean_rel_std_stabilization,lambda1_scaled_high_last3_mean_mean_stabilization,lambda1_scaled_high_last3_mean_std_stabilization,lambda1_scaled_high_last3_mean_rel_std_stabilization,levels_used_for_stabilization,levels_used_for_calibration,status
0,0.923688,0.000681,0.000737,1.939744,0.00143,0.000737,3.064233,0.000205,0.000067,0.167221,0.146451,0.875797,26.261239,1.971942,0.075089,"[3, 4, 5]","[1, 2]",completed
